# Tracking 3D orientation from a real gyroscope: HyperSSM vs. commutative and recurrent baselines

**Task.** An IMU's gyroscope reports angular velocity ω_k in the body frame. The rotation of the body between two
times is the *ordered* product of the small rotations, `Δq = exp(ω_1 Δt/2) ⊗ exp(ω_2 Δt/2) ⊗ ⋯` (quaternions).
Given only the gyroscope readings in a window, predict that relative rotation; ground truth comes from optical
motion capture. Models are trained on **1–2 s windows** and tested on **1 s to 2 min** windows of held-out trials
(up to ~60× the training length).

**Data.** BROAD (Laidig et al., *Data* 2021): 39 trials, one IMU, 286 Hz, gyroscope/accelerometer/magnetometer
and motion-capture orientation, ~170 MB, downloaded from GitHub (MIT / CC-BY-4.0). Trials are split by trial into
train / validation / test, so test motions are never seen. Gyroscope readings are averaged over blocks of 5
samples (57 Hz).

**Models** (same input — the gyroscope reading at each step, scaled — same loss, same training windows):
* **HyperSSM**: one quaternion channel, transition `exp(W ω_k + b)` (12 parameters: a learned calibration of the
  sensor), output the channel's state. 4 numbers of state.
* **ℂ diagonal (selective)**: a Mamba-style selective SSM with 64 complex channels (input-dependent step and
  phase, input term), state read by an MLP. Commutative transitions, 128 numbers of state.
* **LSTM**: 128 hidden units, linear output.
* **Gyro integration** (no learning): the physics formula with the raw readings.
* **Gyro integration, wrong order** (no learning): the same rotations multiplied in reverse order. This is what
  order-blindness costs on real data.
* **No rotation**: always predict the identity, the floor any useful model must beat.

Test trials are the held-out trials that rotate (rotation and combined motions, with and without disturbances).

The loss is `1 − ⟨q̂, q⟩²` on the final step of each window (sign-invariant: q and −q are the same rotation);
the reported error is the rotation angle between prediction and truth, in degrees.

In [ ]:
!pip -q install h5py

In [ ]:
import os, math, time, json, urllib.request
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# ---------------- configuration ----------------
DATA_DIR    = "broad_data"
BASE_URL    = "https://raw.githubusercontent.com/dlaidig/broad/main/data_hdf5/"
DOWNSAMPLE  = 5                  # average gyro over 5 samples: 286 Hz -> 57 Hz
GYRO_SCALE  = 0.5                # network input = ω (rad/s) × GYRO_SCALE
TRAIN_SECS  = (1.0, 2.0)         # training windows
VAL_SECS    = [5.0, 10.0]        # checkpoint selection (beyond the training length)
TEST_SECS   = [1, 2, 5, 10, 30, 60, 120]
N_TEST_WIN  = 256                # windows per test horizon
N_VAL_WIN   = 128
MODELS      = ["HyperSSM", "C-diagonal", "LSTM"]
SEEDS       = [0, 1, 2]
STEPS       = 4_000
BATCH       = 64
LR          = {"HyperSSM": 3e-3, "C-diagonal": 1e-3, "LSTM": 1e-3}
EVAL_EVERY  = 250

# test only on trials that rotate (translation-only trials stay in training: on them, "no rotation" is nearly right)
TEST_TRIALS = ["03_undisturbed_slow_rotation_C", "07_undisturbed_fast_rotation_B",
               "20_undisturbed_slow_combined_360s", "23_undisturbed_fast_combined_360s",
               "24_disturbed_tapping_A", "26_disturbed_phone_vibration_A",
               "31_disturbed_stationary_magnet_D", "38_disturbed_office_B"]
VAL_TRIALS  = ["02_undisturbed_slow_rotation_B", "19_undisturbed_slow_combined_240s",
               "29_disturbed_stationary_magnet_B"]

## Data

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
trials_json = os.path.join(DATA_DIR, "trials.json")
if not os.path.exists(trials_json):
    urllib.request.urlretrieve(BASE_URL + "trials.json", trials_json)
ALL_TRIALS = sorted(json.load(open(trials_json))["trials"].keys())

def load_trial(name):
    path = os.path.join(DATA_DIR, name + ".hdf5")
    if not os.path.exists(path):
        urllib.request.urlretrieve(BASE_URL + name + ".hdf5", path)
    with h5py.File(path, "r") as f:
        gyr, quat, fs = f["imu_gyr"][()], f["opt_quat"][()], float(f.attrs["sampling_rate"])
    n = len(gyr) // DOWNSAMPLE
    gyr = gyr[: n * DOWNSAMPLE].reshape(n, DOWNSAMPLE, 3).mean(1)       # mean angular velocity over each block
    quat = quat[: n * DOWNSAMPLE : DOWNSAMPLE]                            # orientation at the start of each block
    return dict(name=name, gyr=gyr.astype(np.float32), quat=quat, dt=DOWNSAMPLE / fs)

t0 = time.time()
DATA = {name: load_trial(name) for name in ALL_TRIALS}
DT = DATA[ALL_TRIALS[0]]["dt"]
TRAIN_TRIALS = [t for t in ALL_TRIALS if t not in TEST_TRIALS + VAL_TRIALS]
print(f"{len(ALL_TRIALS)} trials loaded in {time.time()-t0:.0f}s; step {DT*1000:.1f} ms; "
      f"train {len(TRAIN_TRIALS)}, val {len(VAL_TRIALS)}, test {len(TEST_TRIALS)}")

def qmul_np(a, b):
    aw, ax, ay, az = np.moveaxis(a, -1, 0); bw, bx, by, bz = np.moveaxis(b, -1, 0)
    return np.stack([aw*bw - ax*bx - ay*by - az*bz, aw*bx + ax*bw + ay*bz - az*by,
                     aw*by - ax*bz + ay*bw + az*bx, aw*bz + ax*by - ay*bx + az*bw], -1)

def conj_np(q): return q * np.array([1, -1, -1, -1])

def make_windows(trials, secs, n, rng):
    """n windows of a fixed duration (or durations drawn from a range), from random trials and start times whose
    motion-capture orientation is valid at both ends. Returns gyro inputs (n, T, 3), lengths, and the target
    relative rotation q(start)^-1 ⊗ q(end) (n, 4)."""
    lo, hi = (secs, secs) if np.isscalar(secs) else secs
    X, Y, L = [], [], []
    while len(X) < n:
        tr = DATA[trials[rng.integers(len(trials))]]
        T = int(round(rng.uniform(lo, hi) / DT))
        if len(tr["gyr"]) <= T + 1: continue
        s = int(rng.integers(0, len(tr["gyr"]) - T - 1))
        q0, q1 = tr["quat"][s], tr["quat"][s + T]
        if np.isnan(q0).any() or np.isnan(q1).any(): continue
        rel = qmul_np(conj_np(q0), q1); rel /= np.linalg.norm(rel)
        X.append(tr["gyr"][s : s + T]); Y.append(rel); L.append(T)
    Tm = max(L); Xp = np.zeros((n, Tm, 3), np.float32)
    for i, x in enumerate(X): Xp[i, : len(x)] = x                       # right-padded; padding = no rotation
    return torch.from_numpy(Xp), torch.tensor(L), torch.from_numpy(np.array(Y, np.float32))

## Models

In [ ]:
def qmul(a, b):
    aw, ax, ay, az = a.unbind(-1); bw, bx, by, bz = b.unbind(-1)
    return torch.stack([aw*bw - ax*bx - ay*by - az*bz, aw*bx + ax*bw + ay*bz - az*by,
                        aw*by - ax*bz + ay*bw + az*bx, aw*bz + ax*by - ay*bx + az*bw], -1)

def qexp(v):
    th = v.norm(dim=-1, keepdim=True)
    return torch.cat([torch.cos(th), torch.sinc(th / math.pi) * v], -1)

def ordered_product(a, right=True):
    """a: (B, T, 4). right=True: a_1 ⊗ a_2 ⊗ ⋯ ⊗ a_T (body-frame composition); else a_T ⊗ ⋯ ⊗ a_1.
    Log-depth tree, as in a parallel scan."""
    B, T, _ = a.shape
    T2 = 1 << max(0, (T - 1).bit_length())
    if T2 > T:
        ident = a.new_zeros(B, T2 - T, 4); ident[..., 0] = 1
        a = torch.cat([a, ident], 1)
    while a.shape[1] > 1:
        a = qmul(a[:, 0::2], a[:, 1::2]) if right else qmul(a[:, 1::2], a[:, 0::2])
    return a[:, 0]

def mask_of(x, L):
    return (torch.arange(x.shape[1], device=x.device)[None] < L[:, None]).float()

class HyperSSMIMU(nn.Module):
    """One quaternion channel. Transition exp(W x_k + b), x_k the scaled gyro reading; state = ordered product.
    (The HyperSSM recurrence q_k = a_k q_{k-1} multiplies on the left; reading out the conjugate, a fixed linear map,
    gives the body-frame order a_1 ⊗ ⋯ ⊗ a_T. Here we compute that product directly.)"""
    def __init__(self):
        super().__init__()
        self.W = nn.Parameter(0.01 * torch.randn(3, 3)); self.b = nn.Parameter(torch.zeros(3))
    def forward(self, x, L):
        v = (x @ self.W.T + self.b) * mask_of(x, L).unsqueeze(-1)
        return ordered_product(qexp(v), right=True)

class CDiagonal(nn.Module):
    """Selective complex diagonal SSM (Mamba-style, commutative transitions):
    h_k = exp(Δ_k(−α + iθ_k)) ⊙ h_{k−1} + Δ_k B x_k, with Δ_k, θ_k input-dependent; MLP readout of h_T."""
    def __init__(self, K=64, hidden=128):
        super().__init__()
        self.dt = nn.Linear(3, K); self.theta = nn.Linear(3, K)
        self.log_alpha = nn.Parameter(torch.log(torch.rand(K) * 0.1 + 1e-3))
        self.B = nn.Linear(3, 2 * K)
        self.head = nn.Sequential(nn.Linear(2 * K, hidden), nn.GELU(), nn.Linear(hidden, 4))
        self.K = K
    def forward(self, x, L, chunk=32):
        outs = []
        for i in range(0, x.shape[0], chunk):                     # chunked: the closed form below is O(T·K) memory
            xb, Lb = x[i:i + chunk], L[i:i + chunk]
            m = mask_of(xb, Lb)
            d = F.softplus(self.dt(xb)) * m.unsqueeze(-1)          # (B, T, K); padding: Δ = 0, so a = 1 and no input
            log_a = torch.complex(-d * self.log_alpha.exp(), d * self.theta(xb))
            Bx = torch.view_as_complex(self.B(xb).view(*xb.shape[:2], self.K, 2).contiguous()) * d
            S = torch.cumsum(log_a, 1)                             # h_T = Σ_k exp(S_T − S_k) Δ_k B x_k
            h = (torch.exp(S[:, -1:] - S) * Bx).sum(1)
            outs.append(self.head(torch.cat([h.real, h.imag], -1)))
        return F.normalize(torch.cat(outs), dim=-1)

class LSTMIMU(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.lstm = nn.LSTM(3, hidden, batch_first=True); self.head = nn.Linear(hidden, 4)
    def forward(self, x, L):
        out, _ = self.lstm(x)
        return F.normalize(self.head(out[torch.arange(len(L)), L - 1]), dim=-1)

def gyro_integration(x, L, wrong_order=False):
    """x: raw gyro readings (rad/s)."""
    v = x * DT / 2 * mask_of(x, L).unsqueeze(-1)
    return ordered_product(qexp(v), right=not wrong_order)

MAKE = {"HyperSSM": HyperSSMIMU, "C-diagonal": CDiagonal, "LSTM": LSTMIMU}

## Training and evaluation

In [ ]:
def angle_deg(qp, qt):
    return torch.rad2deg(2 * torch.arccos(torch.clamp((qp * qt).sum(-1).abs(), max=1.0)))

@torch.no_grad()
def errors(fn, windows, bs=64):
    x, L, y = windows; out = []
    for i in range(0, len(L), bs):
        out.append(angle_deg(fn(x[i:i+bs].to(DEVICE), L[i:i+bs].to(DEVICE)), y[i:i+bs].to(DEVICE)).cpu())
    return torch.cat(out).numpy()

def scaled(fn):
    return lambda x, L: fn(x * GYRO_SCALE, L)

val_sets = {s: make_windows(VAL_TRIALS, s, N_VAL_WIN, np.random.default_rng(100 + int(s))) for s in VAL_SECS}
test_sets = {s: make_windows(TEST_TRIALS, float(s), N_TEST_WIN, np.random.default_rng(200 + int(s))) for s in TEST_SECS}

def train(name, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(1_000 + seed)
    model = MAKE[name]().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR[name], weight_decay=0.0)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, STEPS)
    best, best_state, t0 = float("inf"), None, time.time()
    for step in range(1, STEPS + 1):
        x, L, y = make_windows(TRAIN_TRIALS, TRAIN_SECS, BATCH, rng)
        x, L, y = (x * GYRO_SCALE).to(DEVICE), L.to(DEVICE), y.to(DEVICE)
        loss = (1 - (model(x, L) * y).sum(-1) ** 2).mean()
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if step % EVAL_EVERY == 0:
            model.eval()
            val = float(np.mean([np.median(errors(scaled(model), val_sets[s])) for s in VAL_SECS]))
            model.train()
            if val < best:
                best, best_state = val, {k: v.detach().clone() for k, v in model.state_dict().items()}
            if step % (4 * EVAL_EVERY) == 0:
                print(f"  {name:10s} seed {seed} step {step:5d} loss {loss.item():.5f} "
                      f"val median error {val:6.2f}° (best {best:6.2f}°) [{time.time()-t0:.0f}s]", flush=True)
    model.load_state_dict(best_state); model.eval()
    return model

In [ ]:
results = {}
results["gyro integration"] = {s: errors(lambda x, L: gyro_integration(x, L), test_sets[s]) for s in TEST_SECS}
results["gyro, wrong order"] = {s: errors(lambda x, L: gyro_integration(x, L, True), test_sets[s]) for s in TEST_SECS}
ident = lambda x, L: torch.tensor([1.0, 0, 0, 0], device=x.device).expand(len(L), 4)
results["no rotation"] = {s: errors(ident, test_sets[s]) for s in TEST_SECS}
trained = {}
for name in MODELS:
    for seed in SEEDS:
        print(f"{name}, seed {seed}")
        m = train(name, seed); trained[(name, seed)] = m
        results[f"{name}|{seed}"] = {s: errors(scaled(m), test_sets[s]) for s in TEST_SECS}
        med = [np.median(results[f'{name}|{seed}'][s]) for s in TEST_SECS]
        print("  test median error (°) at " + ", ".join(f"{s}s: {e:.1f}" for s, e in zip(TEST_SECS, med)))

## Results: median rotation error on held-out trials vs. window length

In [ ]:
def summary(key_prefix):
    keys = [k for k in results if k == key_prefix or k.startswith(key_prefix + "|")]
    meds = np.array([[np.median(results[k][s]) for s in TEST_SECS] for k in keys])
    return meds.mean(0), meds.std(0)

rows = ["no rotation", "gyro integration", "gyro, wrong order"] + MODELS
print(f"median error (°) on held-out trials; mean ± s.d. over {len(SEEDS)} seeds; trained on {TRAIN_SECS[0]:.0f}–{TRAIN_SECS[1]:.0f} s")
print(f"{'model':20s}" + "".join(f"{str(s)+' s':>13s}" for s in TEST_SECS))
for r in rows:
    mu, sd = summary(r)
    print(f"{r:20s}" + "".join(f"{m:7.1f}±{d:<5.1f}" if d > 0 else f"{m:13.1f}" for m, d in zip(mu, sd)))

fig, ax = plt.subplots(figsize=(6.4, 4.2))
style = {"no rotation": dict(color="0.75", ls="-."), "gyro integration": dict(color="0.4", ls="--"), "gyro, wrong order": dict(color="0.6", ls=":"),
         "HyperSSM": dict(color="#2456a6", lw=2.2), "C-diagonal": dict(color="#d97706"), "LSTM": dict(color="#7c3aed")}
for r in rows:
    mu, sd = summary(r)
    ax.plot(TEST_SECS, mu, marker="o", ms=3.5, label=r, **style[r])
    if sd.any(): ax.fill_between(TEST_SECS, mu - sd, mu + sd, alpha=0.15, color=style[r]["color"])
ax.axvspan(TRAIN_SECS[0], TRAIN_SECS[1], color="0.85", alpha=0.6, lw=0, label="training lengths")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("window length (s)"); ax.set_ylabel("median rotation error (°)")
ax.set_title("Orientation from a gyroscope, held-out trials (BROAD)")
ax.legend(fontsize=8, frameon=False); plt.tight_layout(); plt.savefig("imu_orientation.png", dpi=200); plt.show()

## What HyperSSM learned: the sensor's calibration
Ideal gyro integration corresponds to `W = (Δt/2)/GYRO_SCALE · I` and `b = 0`. The learned W and b are the
sensor's scale/misalignment and bias, read straight off the 12 parameters.

In [ ]:
m = trained[("HyperSSM", SEEDS[0])]
ideal = DT / 2 / GYRO_SCALE
print("learned W / ideal:\n", np.round(m.W.detach().cpu().numpy() / ideal, 4))
print("learned bias (rad/s):", np.round(-m.b.detach().cpu().numpy() / (ideal * GYRO_SCALE), 5),
      "(the correction subtracted from the gyro reading)")